In [31]:
#здесь мы соберём информацию по зарплате каждой должности по регионам с помощью написанного парсера

In [32]:
#просто импорт необходимых штук
import requests
import pandas as pd

def get_hh_vacancies(key_words, area, pages = 5, only_with_salary = True):
#пояснения к параметрам функции:
#это параметры к api
#key_words - ключевые слова, по которым ищем вакансию
#area - регион, где мы ищем вакансию
#pages - количество страниц, которых мы хотим просмотреть(значение по умолчанию = 5)
#only_with_salary - фильтр для тех вакансий, где явно указана зарплата(значение по умолчанию = True)
    base_url = "https://api.hh.ru/vacancies" # базовый API HH.ru

    vacancies = [] #сюда будем записывать данные

    for page in range(pages):
        params = {"text": key_words, "area": area, "page": page, "only_with_salary": only_with_salary}
        response = requests.get(base_url, params=params)#получили наш HTTP-ответ от API

        if response.status_code != 200:
          print(f"Ошибка! Причина: {response.text}")  #проверяет, что запрос произршёл успешно
          break

        data = response.json()#переводим ответ в формат json, чтобы дальше извлечь необохимые данные
        vacancies.extend(data.get("items", []))
        #data.get("items", []) возвращает значение для ключа "items"
        #метод extend добавляет в конец списка информацию по каждой вакансии
        #то есть фактически vacancies сейчас представляет собой, список словарей
        #где в каждом словаре записаны характеристики каждой вакансии

        if page >= data["pages"] - 1: #проверка на достижение последней страницы
            break

    results = []
    for vacancy in vacancies:#проходимся по каждой вакансии
        salary = vacancy.get("salary") #достаём содержимое в ячейке salary
        #поскольку зачастую работодатели указывают вилку зп, а не конкретное значение,
        #то это надо учитывать
        real_salary = 0
        if salary:
          if salary["from"] and salary["to"]: #если указаны обе границы, то берём среднее арифм.
            real_salary = (salary["from"] + salary["to"]) / 2
          else:#если указана лишь одна из ограниц, берём её
            real_salary = salary["from"] or salary["to"]


        results.append({
            "Вакансия": vacancy["name"],
            "Зарплата": real_salary,
            "Регион": area
        })

    return pd.DataFrame(results)


In [33]:
cities = {'Город':['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань'], 'ID':[1, 2, 4, 3, 88]}
#это мы тоже получили с помощью api hh ru

In [34]:
#понятно, что при парсинге будут выбросы. Например где должность на самом деле не зоотехник, а просто как-то связана с ней
#и в вакансии указано ключевое слово, на таких должностях либо сильно больше зп, либо сильно меньше
#их надо отфильтровать
#воспользуемся методом IQR(межквартильного размаха)
def Filter(df):
  Q1 = df['Зарплата'].quantile(0.25)
  Q3 = df['Зарплата'].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  result_df = df[(df['Зарплата'] >= lower_bound) & (df['Зарплата'] <= upper_bound)]
  result_df.reset_index(drop=True, inplace=True)
  return result_df

In [69]:
#сначала рассмотрим Москву
#поскольку парсер ищет вакансии по определённым словам, то логично было бы давать на вход список возможных ключевых слов
#поэтому напишем цикл, который проходится по всем ключевым фразам и соединяет таблички в исходную
#запихнём это всё функцию
def get_salary(profi_words, area, pages):
  df_total = pd.DataFrame()
  for word in profi_words:
    df_prof = get_hh_vacancies(word, area, pages)
    df_total = pd.concat([df_total, df_prof], axis=0, ignore_index=True)

  filtered_df_total = Filter(df_total)
  average_salary = filtered_df_total['Зарплата'].mean()
  return average_salary


In [95]:
#поскольку у нас 5 городов и 5 должностей, запихнём всё это снова в ещё один цикл
zoo_prof = ['зоотехник', 'зоотехник-технолог', 'зоотехник-селекционер', 'зоотехник ферма']
vet_prof = ['ветеринар']
driver_prof = ['водитель форд транзит','водитель газель']
eng_prof = ['механик ппр']
worker_prof = ['разнорабочий']
professions = [zoo_prof, vet_prof, driver_prof, eng_prof, worker_prof]
cities_id = [1,2,4,3,88]
#мск, спб, новосиб, ект, казань
prof_groups = ['zoo', 'vet', 'driver', 'eng', 'worker']
df = pd.DataFrame(index=cities_id, columns=prof_groups)
for id in cities_id:
  for i, prof in enumerate(professions):
    salary = round(get_salary(prof, id, 15),2)
    df.loc[id, prof_groups[i]] = salary
df

,zoo,vet,driver,eng,worker
1,115238.1,86150.34,117413.37,116107.54,104519.45
2,130142.86,70492.14,105608.58,110447.85,86803.31
4,90500.0,77681.0,78431.88,178837.5,63036.12
3,115500.0,75888.11,77693.62,146574.07,66699.84
88,99892.31,67655.11,88264.52,159314.29,75189.79


In [96]:
#сделаем норм индексы
new_indexes = ['Москва','Санкт-Петербург','Новосибирск','Екатеринбург', 'Казань']
df.index = new_indexes
df

,zoo,vet,driver,eng,worker
Москва,115238.1,86150.34,117413.37,116107.54,104519.45
Санкт-Петербург,130142.86,70492.14,105608.58,110447.85,86803.31
Новосибирск,90500.0,77681.0,78431.88,178837.5,63036.12
Екатеринбург,115500.0,75888.11,77693.62,146574.07,66699.84
Казань,99892.31,67655.11,88264.52,159314.29,75189.79


In [92]:
#вот мы и получили таблицу со средними зарплатами по каждой профессии в 5 крупнейших городах России

In [93]:
#теперь, имея зарплаты, можем оценить издержки фирмы
#также имеем цену на куриц в Москве
import random
ch_prices = [900.0, 530.0, 600.0, 550.0, 540.0]
ch_price = sum(ch_prices) / len(ch_prices)
k
m
t
x
y
z
q
w
u
p

def costs(farms_in_use, new_farms):
  sick_chicks = random.randint(0, (farms_in_use + new_farms) * 2) #от количества больных курочек зависят траты на ветеринара
  #считаем, что оптимальное количество куриц - 2 штуки на одну мобильную ферму
  eng_expenditure = new_farms*k*df['Москва']['eng'] + farms_in_use*m*df['Москва']['eng']
  vet_expenditure = sick_chicks*t*df['Москва']['vet']
  driver_expenditure = new_farms*x*df['Москва']['driver'] + farms_in_use*y*df['Москва']['driver'] + sick_chicks*z*df['Москва']['driver']
  worker_expenditure = farms_in_use*q*df['Москва']['worker'] + new_farms*w*df['Москва']['worker']
  zoo_expenditure = farms_in_use*u*df['Москва']['zoo'] + new_farms*p*df['Москва']['zoo'] + sick_chicks*a*df['Москва']['zoo']

  return eng_expenditure + vet_expenditure + driver_expenditure + worker_expenditure + zoo_expenditure



624.0